# Model Comparison Summary
Notebook นี้รวบรวมผลจากไฟล์ metrics ที่สร้างโดย notebook เทรนแต่ละโมเดล แล้ว export ตารางสำหรับ Results.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = os.path.abspath('..')
reports_dir = os.path.join(project_root, 'reports', 'model_comparison')
models_dir = os.path.join(project_root, 'models')
os.makedirs(reports_dir, exist_ok=True)

metric_files = [
    os.path.join(reports_dir, 'svm_metrics.json'),
    os.path.join(reports_dir, 'lstm_gru_metrics.json'),
    os.path.join(reports_dir, 'stgcn_metrics.json')
]

rows = []
missing = []
for path in metric_files:
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            rows.append(json.load(f))
    else:
        missing.append(path)

def load_latest_mlp_metadata(models_root):
    version_dirs = []
    for name in os.listdir(models_root):
        full = os.path.join(models_root, name)
        if os.path.isdir(full) and name.startswith('v_'):
            version_dirs.append(full)
    version_dirs.sort(reverse=True)

    for vd in version_dirs:
        metadata_path = os.path.join(vd, 'model_metadata.json')
        if not os.path.exists(metadata_path):
            continue
        with open(metadata_path, 'r', encoding='utf-8') as f:
            m = json.load(f)

        acc = m.get('test_accuracy', np.nan)
        if isinstance(acc, (int, float)) and acc > 1:
            acc = acc / 100.0

        return {
            'model': 'Our Proposed Model (MLP 128-64)',
            'accuracy': float(acc) if not pd.isna(acc) else np.nan,
            'macro_f1': np.nan,
            'inference_latency_ms': np.nan,
            'train_samples': np.nan,
            'test_samples': np.nan,
            'num_features': m.get('total_features', np.nan),
            'source': metadata_path
        }

    return None

mlp_row = load_latest_mlp_metadata(models_dir)
if mlp_row is not None:
    rows.append(mlp_row)

if not rows:
    raise RuntimeError('No metrics found. Run training notebooks first.')

df = pd.DataFrame(rows)

for col in ['accuracy', 'macro_f1', 'inference_latency_ms']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values(by='accuracy', ascending=False).reset_index(drop=True)

print('Loaded models:', list(df['model']))
if missing:
    print('\nMissing metric files:')
    for m in missing:
        print('-', m)

df

In [ ]:
df_export = df.copy()
df_export['accuracy_percent'] = (df_export['accuracy'] * 100).round(2)
if 'macro_f1' in df_export.columns:
    df_export['macro_f1'] = df_export['macro_f1'].round(4)
if 'inference_latency_ms' in df_export.columns:
    df_export['inference_latency_ms'] = df_export['inference_latency_ms'].round(4)

out_csv = os.path.join(reports_dir, 'model_comparison_table.csv')
out_md = os.path.join(reports_dir, 'model_comparison_table.md')

df_export.to_csv(out_csv, index=False, encoding='utf-8-sig')
with open(out_md, 'w', encoding='utf-8') as f:
    f.write(df_export.to_markdown(index=False))

print('Saved:', out_csv)
print('Saved:', out_md)

In [ ]:
plot_df = df.copy()
plot_df['accuracy_percent'] = plot_df['accuracy'] * 100

plt.figure(figsize=(10, 5))
plt.bar(plot_df['model'], plot_df['accuracy_percent'])
plt.ylabel('Accuracy (%)')
plt.title('Model Accuracy Comparison')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

latency_df = plot_df.dropna(subset=['inference_latency_ms'])
if not latency_df.empty:
    plt.figure(figsize=(10, 5))
    plt.bar(latency_df['model'], latency_df['inference_latency_ms'])
    plt.ylabel('Latency (ms / sample)')
    plt.title('Inference Latency Comparison')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()